# OCTRON rendering (`render` in cli)

This notebook shows the simplest Python-first way to turn OCTRON prediction output into shareable videos, without opening the GUI.

It uses `octron.tools.render.run_render()`, the same high-level function used by the command line (`octron render`). Two kinds of output are covered:

- **Overview (overlay)**: the original video with masks, bounding boxes and labels drawn on top.
- **Tracklets**: one stabilised crop video per tracked animal, centred on its trajectory.

You need a prediction output directory produced by `octron predict` first — see `OCTRON_YOLO_predict_simple.ipynb`.


## 1. Imports

`run_render()` reads a prediction output folder (masks + tracks + metadata) and writes rendered `.mp4` files.
`report_bbox_sizes()` is a small helper that reports per-track bounding-box sizes, useful for choosing a tracklet crop size.


In [ ]:
from pathlib import Path
from octron.tools.render import run_render, report_bbox_sizes

## 2. Configure paths

`predictions_path` is a single prediction output directory created by `octron predict`. It looks like:

```
.../octron_predictions/<video>_<tracker>/
```

The original video is auto-detected when it sits next to the `octron_predictions/` folder. If it cannot be found, set `video_path` explicitly.


In [ ]:
# A single prediction output directory (octron_predictions/<video>_<tracker>/).
predictions_path = Path("/Users/horst/Downloads/tomopteris-planktonis-3/test/octron_predictions/example_video_bytetrack")

# Original video. None = auto-detect if it sits next to octron_predictions/.
video_path = None
# video_path = Path("path/to/myvideo.mp4")

# Output directory. None writes into <predictions_path>/rendered/.
output_path = None
# output_path = Path("path/to/rendered_output")


## 3. Render an overview (overlay)

With `tracklets=False` (the default), `run_render()` draws masks, boxes and labels on top of the original video.

`preset` trades resolution for speed/size:

- `"preview"` — 0.25× resolution (fastest)
- `"draft"` — 0.5× resolution (default)
- `"final"` — full resolution

This mirrors the CLI command:

```bash
octron render PREDICTIONS_PATH --preset draft
```

The output is written as `overlay_<preset>.mp4` in the output directory.


In [ ]:
run_render(
    predictions_path=predictions_path,
    video_path=video_path,
    output_path=output_path,
    preset="final",          # "preview" | "draft" | "final"
    draw_masks=True,
    draw_boxes=True,
    draw_labels=True,
    alpha=0.4,              # mask overlay opacity (0-1)
    min_confidence=0.5,      # skip detections below this confidence
)


### Useful overlay options

A few extra knobs you can pass to the overlay render:

- `start` / `end`: render only a frame range (e.g. a short preview).
- `track_ids`: render only specific tracks, e.g. `"1,3"` or `[1, 3]`.
- `min_observations`: skip tracks with very few detections.
- `skip_empty`: drop frames that have no detection (handy if you predicted with `skip_frames`).


In [ ]:
# Example: a 200-frame preview of selected tracks only.
run_render(
    predictions_path=predictions_path,
    video_path=video_path,
    output_path=output_path,
    preset="final",
    start=1900,
    end=4000,
    track_ids=24,       # or [1, 3] or "1,3"
    min_confidence=0.5,
    skip_empty=True,
)


## 4. Render tracklets (one crop video per animal)

Tracklets are per-animal crop videos centred on each track's smoothed trajectory.

First, inspect the bounding-box sizes so you can pick a sensible crop size. If you skip this, `tracklet_size=None` auto-computes a size from the largest bounding box.


In [ ]:
stats = report_bbox_sizes(predictions_path)


Now render the tracklets with `tracklets=True`. This mirrors:

```bash
octron render PREDICTIONS_PATH --tracklets
```

Each animal is written as `tracklet_<label>_track<id>_<preset>.mp4`.


In [ ]:
run_render(
    predictions_path=predictions_path,
    video_path=video_path,
    output_path=output_path,
    tracklets=True,
    tracklet_size=None,            # None = auto (largest bbox + padding); or e.g. 200
    tracklet_smooth_sigma=2.0,     # centroid smoothing in frames (0 = off)
    min_confidence=0.5,
    skip_empty=True,               # drop frames where the animal was not detected
    preset="final",
)


### Useful tracklet options

- `tracklet_size`: fix the crop side length in pixels (use `report_bbox_sizes` to choose).
- `tracklet_interpolate_limit`: bridge short gaps in the trajectory (max N missing frames).
- `tracklet_segment_only`: black out everything outside the animal's segmentation mask.
- `tracklet_offset`: shift the crop centre by `(dx, dy)` in source-video pixels.
- `also_overlay`: draw masks/boxes onto the crops (useful to check centroid placement).
- `trim`: trim each crop video to the track's first and last observation.


In [ ]:
# Example: fixed-size, mask-only tracklets that bridge short detection gaps.
# run_render(
#     predictions_path=predictions_path,
#     video_path=video_path,
#     output_path=output_path,
#     tracklets=True,
#     tracklet_size=200,
#     tracklet_smooth_sigma=2.0,
#     tracklet_interpolate_limit=5,
#     tracklet_segment_only=True,
#     also_overlay=False,
#     trim=True,
#     min_confidence=0.5,
# )


## 5. Where the files go

By default everything is written under:

```
<predictions_path>/rendered/
    overlay_<preset>.mp4                      # overview (overlay) render
    tracklet_<label>_track<id>_<preset>.mp4   # one file per tracked animal
```

Pass `output_path` to write elsewhere.